# Traveling Salesman Problem (TSP)

A salesperson starts in **NYC**, must visit every city exactly once, and return to NYC, traveling the minimum total distance.

In [ ]:
# Define packages
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

In [ ]:
# DISTANCE data
NYC, PHI, DC, BOS = 'NYC', 'PHI', 'DC', 'BOS'
DEPOT = NYC

distance_data = pd.DataFrame(columns=['From', 'To', 'Distance'])
distance_data['From']     = [NYC, NYC, NYC, NYC, PHI, PHI, PHI, PHI, DC, DC, DC, DC, BOS, BOS, BOS, BOS]
distance_data['To']       = [NYC, PHI, DC, BOS, NYC, PHI, DC, BOS, NYC, PHI, DC, BOS, NYC, PHI, DC, BOS]
distance_data['Distance'] = [0, 97, 229, 215, 97, 0, 142, 308, 229, 142, 0, 443, 215, 308, 443, 0]
distance_data

In [ ]:
# Create empty model
m = AbstractModel()

**Sets and parameters**

- $V$: set of cities
- $A = \{(i,j) \in V \times V : i \ne j\}$: set of arcs (no self-loops)
- $c_{ij}$: distance from city $i$ to city $j$
- $n = |V|$: number of cities

In [ ]:
# Define Sets
m.V = Set()
m.A = Set(within=m.V * m.V, initialize=lambda m: [(i, j) for i in m.V for j in m.V if i != j])

# Define Params
m.c = Param(m.V, m.V)
m.n = Param(initialize=lambda m: len(m.V))

**Decision variables**

- $x_{ij} \in \{0,1\}$: 1 if the tour travels arc $i \to j$
- $u_i$: position of city $i$ in the tour (used by the MTZ subtour-elimination constraints)

$1 \le u_i \le n-1 \qquad \forall i \in V \setminus \{NYC\}$

In [ ]:
# Define Decision Vars
m.x = Var(m.A, domain=Binary)
m.u = Var(m.V, domain=NonNegativeIntegers, bounds=lambda m, i: (0, 0) if i == DEPOT else (1, m.n - 1))

**Leave each city exactly once**

$\sum_{j \in V,\, j \ne i} x_{ij} = 1 \qquad \forall i \in V$

In [ ]:
def Leave_Once_Rule(m, i):
    return sum(m.x[i, j] for j in m.V if j != i) == 1
m.LeaveOnce = Constraint(m.V, rule=Leave_Once_Rule)

**Enter each city exactly once**

$\sum_{i \in V,\, i \ne j} x_{ij} = 1 \qquad \forall j \in V$

In [ ]:
def Enter_Once_Rule(m, j):
    return sum(m.x[i, j] for i in m.V if i != j) == 1
m.EnterOnce = Constraint(m.V, rule=Enter_Once_Rule)

**Subtour elimination (Miller–Tucker–Zemlin)**

If the tour goes from $i$ to $j$, then $j$ must come later in the tour than $i$ ($u_j \ge u_i + 1$). Otherwise the constraint is not binding.

$u_i - u_j + (n-1)\, x_{ij} \le n-2 \qquad \forall (i,j) \in A,\; i, j \ne NYC$

In [ ]:
def MTZ_Rule(m, i, j):
    if i == DEPOT or j == DEPOT:
        return Constraint.Skip
    return m.u[i] - m.u[j] + (m.n - 1) * m.x[i, j] <= m.n - 2
m.MTZ = Constraint(m.A, rule=MTZ_Rule)

**Minimize total distance**

$\min \sum_{(i,j) \in A} c_{ij}\, x_{ij}$

In [ ]:
def Min_Distance_Rule(m):
    return sum(m.c[i, j] * m.x[i, j] for (i, j) in m.A)
m.Min_Distance = Objective(rule=Min_Distance_Rule, sense=minimize)

In [ ]:
# Build and read instance data
instanceData = {None: {
    'V': {None: list(distance_data['From'].unique())},
    'c': distance_data.set_index(['From', 'To'])['Distance'].to_dict(),
}}
instance = m.create_instance(instanceData)

In [ ]:
# Solve instance
solver = SolverFactory('glpk')
solution = solver.solve(instance)
print('Solver Status:')
print(solution['Solver'])
print('Optimum Objective Function Value:')
print(value(instance.Min_Distance))

In [ ]:
# Create output data: follow the arcs from the depot to get the tour in order
next_city = {i: j for (i, j) in instance.A if value(instance.x[i, j]) > 0.5}

tour = [DEPOT]
while next_city[tour[-1]] != DEPOT:
    tour.append(next_city[tour[-1]])
tour.append(DEPOT)

output_route = pd.DataFrame(
    [{'Stop': k + 1, 'From': i, 'To': j, 'Distance': instance.c[i, j]}
     for k, (i, j) in enumerate(zip(tour[:-1], tour[1:]))]
)

print('===== OPTIMAL TOUR =====')
print(' → '.join(tour))
print(f'Total Distance: {value(instance.Min_Distance):.0f}')
output_route